# How quickly are changes in crude oil inventories passed on to pump prices?

**Case study:** U.S. crude oil market after the Iran conflict (Feb 28, 2026).

Data source: U.S. Energy Information Administration (EIA) — Weekly series.

## 1. Setup

In [ ]:
import os
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from matplotlib.ticker import FuncFormatter
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("EIA_API_KEY")

if not API_KEY:
    raise ValueError("EIA_API_KEY not found. Add it to your .env file.")

START_DATE = "2026-02-28"
EVENT_DATE = pd.Timestamp("2026-02-28")

## 2. Helper function for paginated requests

In [ ]:
def fetch_eia_data(url, params, page_size=5000):
    all_rows = []
    offset = 0
    while True:
        params["offset"] = offset
        params["length"] = page_size
        r = requests.get(url, params=params).json()
        if "response" not in r:
            raise RuntimeError(f"API error: {r}")
        rows = r["response"]["data"]
        if not rows:
            break
        all_rows.extend(rows)
        if len(rows) < page_size:
            break
        offset += page_size
    df = pd.DataFrame(all_rows)
    df["period"] = pd.to_datetime(df["period"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    return df

## 3. Fetch crude oil inventories

In [ ]:
url_inv = "https://api.eia.gov/v2/petroleum/stoc/wstk/data/"

params_inv = {
    "api_key": API_KEY,
    "frequency": "weekly",
    "data[0]": "value",
    "facets[duoarea][]": "NUS",
    "facets[product][]": "EPC0",
    "start": START_DATE,
    "sort[0][column]": "period",
    "sort[0][direction]": "desc",
}

df_inv = fetch_eia_data(url_inv, params_inv)

SERIE_INV = "U.S. Ending Stocks excluding SPR of Crude Oil (Thousand Barrels)"
df_inv = (
    df_inv[df_inv["series-description"] == SERIE_INV]
    .drop_duplicates(subset="period")
    .sort_values("period")
    .reset_index(drop=True)
)

print(f"Rows: {len(df_inv)}")
df_inv.head()

## 4. Fetch retail gasoline prices

In [ ]:
url_pre = "https://api.eia.gov/v2/petroleum/pri/gnd/data/"

params_pre = {
    "api_key": API_KEY,
    "frequency": "weekly",
    "data[0]": "value",
    "facets[duoarea][]": "NUS",
    "facets[product][]": "EPMR",
    "start": START_DATE,
    "sort[0][column]": "period",
    "sort[0][direction]": "desc",
}

df_pre = fetch_eia_data(url_pre, params_pre)
df_pre = (
    df_pre.drop_duplicates(subset="period")
    .sort_values("period")
    .reset_index(drop=True)
)

print(f"Rows: {len(df_pre)}")
df_pre.head()

## 5. Merge by week

In [ ]:
inv = df_inv[["period", "value"]].rename(columns={"value": "inventory_kbbl"})
pre = df_pre[["period", "value"]].rename(columns={"value": "price_usd_gal"})

inv["week"] = inv["period"].dt.to_period("W")
pre["week"] = pre["period"].dt.to_period("W")

df_merged = (
    pd.merge(
        inv[["week", "inventory_kbbl"]],
        pre[["week", "price_usd_gal"]],
        on="week",
        how="inner",
    )
    .sort_values("week")
    .reset_index(drop=True)
)
df_merged["date"] = df_merged["week"].dt.start_time
df_merged

## 6. Time series visualization

In [ ]:
sns.set_theme(style="whitegrid", context="notebook")

inv_color = "#1f4e79"
price_color = "#c1272d"
event_color = "#2c2c2c"
band_color = "#fff4e6"

inv_start, inv_end = df_merged["inventory_kbbl"].iloc[0], df_merged["inventory_kbbl"].iloc[-1]
inv_change = (inv_end - inv_start) / inv_start * 100
inv_change_abs = inv_end - inv_start

pre_start, pre_end = df_merged["price_usd_gal"].iloc[0], df_merged["price_usd_gal"].iloc[-1]
pre_change = (pre_end - pre_start) / pre_start * 100
pre_change_abs = pre_end - pre_start

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True, gridspec_kw={"hspace": 0.25})

for ax in axes:
    ax.axvspan(EVENT_DATE, df_merged["date"].max() + pd.Timedelta(days=4),
               color=band_color, alpha=0.6, zorder=0)
    ax.axvline(EVENT_DATE, color=event_color, linestyle="--", linewidth=1.4, alpha=0.7, zorder=1)

ax1, ax2 = axes

sns.lineplot(data=df_merged, x="date", y="inventory_kbbl", ax=ax1,
             color=inv_color, linewidth=2.8, marker="o", markersize=10,
             markerfacecolor="white", markeredgewidth=2.2, zorder=3)
ax1.set_ylabel("Commercial Crude Inventory\n(thousand barrels)", fontsize=11, fontweight="medium")
ax1.set_title("Inventory Buildup", loc="left", fontsize=13, fontweight="bold", color=inv_color, pad=10)
ax1.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:,.0f}"))

change_color = "#27ae60" if inv_change > 0 else "#c0392b"
ax1.annotate(f"{inv_change:+.1f}%\n({inv_change_abs:+,.0f} kbbl)",
             xy=(df_merged["date"].iloc[-1], inv_end),
             xytext=(75, -5), textcoords="offset points",
             fontsize=12, fontweight="bold", color=change_color, ha="left", va="center",
             bbox=dict(boxstyle="round,pad=0.5", facecolor="white", edgecolor=change_color, linewidth=1.5))

sns.lineplot(data=df_merged, x="date", y="price_usd_gal", ax=ax2,
             color=price_color, linewidth=2.8, marker="o", markersize=10,
             markerfacecolor="white", markeredgewidth=2.2, zorder=3)
ax2.set_ylabel("Regular Gasoline Price\n(USD/gallon)", fontsize=11, fontweight="medium")
ax2.set_xlabel("Date", fontsize=11, fontweight="medium")
ax2.set_title("Price Surge", loc="left", fontsize=13, fontweight="bold", color=price_color, pad=10)
ax2.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"${y:.2f}"))

change_color = "#c0392b" if pre_change > 0 else "#27ae60"
ax2.annotate(f"{pre_change:+.1f}%\n(+${pre_change_abs:.2f})",
             xy=(df_merged["date"].iloc[-1], pre_end),
             xytext=(75, -5), textcoords="offset points",
             fontsize=12, fontweight="bold", color=change_color, ha="left", va="center",
             bbox=dict(boxstyle="round,pad=0.5", facecolor="white", edgecolor=change_color, linewidth=1.5))

ymin, ymax = ax2.get_ylim()
ax2.text(EVENT_DATE, ymax - (ymax - ymin) * 0.05,
         "  Iran conflict starts\n  Feb 28, 2026",
         fontsize=9.5, color=event_color, fontweight="medium",
         va="top", ha="left", style="italic")

ax2.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))

for ax in axes:
    sns.despine(ax=ax)
    ax.tick_params(labelsize=10)
    ax.margins(x=0.08)

fig.suptitle("U.S. Crude Oil Market — Iran Conflict Impact",
             fontsize=16, fontweight="bold", y=0.995)
fig.text(0.5, 0.955,
         f"Weekly data, {df_merged['date'].iloc[0].strftime('%b %d')} – "
         f"{df_merged['date'].iloc[-1].strftime('%b %d, %Y')}",
         ha="center", fontsize=10.5, color="#555", style="italic")
fig.text(0.5, 0.005,
         "Source: U.S. Energy Information Administration (EIA)  •  Series: stoc/wstk and pri/gnd",
         ha="center", fontsize=9, style="italic", color="gray")

plt.tight_layout(rect=[0, 0.02, 1, 0.94])
plt.savefig("output/timeseries_dual_panel.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Correlation analysis

In [ ]:
df_plot = df_merged[["date", "inventory_kbbl", "price_usd_gal"]].reset_index(drop=True)

fig, ax = plt.subplots(figsize=(9, 6))

sns.scatterplot(data=df_plot, x="inventory_kbbl", y="price_usd_gal",
                ax=ax, s=150, color="#6a3d9a", alpha=0.8,
                edgecolor="white", linewidth=2)

sns.regplot(data=df_plot, x="inventory_kbbl", y="price_usd_gal",
            ax=ax, scatter=False, color="black",
            line_kws={"linewidth": 2, "linestyle": "--"}, ci=95)

for _, row in df_plot.iterrows():
    ax.annotate(row["date"].strftime("%b %d"),
                xy=(row["inventory_kbbl"], row["price_usd_gal"]),
                xytext=(8, 8), textcoords="offset points",
                fontsize=9, color="#444")

corr = df_plot["inventory_kbbl"].corr(df_plot["price_usd_gal"])

ax.set_xlabel("Commercial Crude Inventory (thousand bbl)", fontsize=11, fontweight="medium")
ax.set_ylabel("Regular Gasoline Price (USD/gallon)", fontsize=11, fontweight="medium")
ax.set_title(f"Inventory vs Gasoline Price — Post Iran Conflict\nPearson correlation: {corr:.3f}",
             fontsize=13, fontweight="bold", pad=15)
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"${y:.2f}"))

fig.text(0.5, -0.02,
         "Source: U.S. Energy Information Administration (EIA) — Weekly Data",
         ha="center", fontsize=9, style="italic", color="gray")

sns.despine()
plt.tight_layout()
plt.savefig("output/scatter_correlation.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Export merged dataset

In [ ]:
df_merged.to_csv("output/oil_inventories_vs_gas_prices.csv", index=False)
print("Saved to output/oil_inventories_vs_gas_prices.csv")